In [3]:
# ===============================
# 🚀 LoRA Merge + ValueHead + Test
# ===============================


# ✅ Imports
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from trl import AutoModelForCausalLMWithValueHead

# === Configurações ===
LORA_REPO = "augustocsc/Se124M100KInfPrompt_EOS"
BASE_MODEL = "gpt2"
OUTPUT_DIR = "./modelo_final_para_ppo"
MODEL_HUB = "augustocsc/Se124M100KInfPrompt_EOS_Merged"
# === Carregar o tokenizer correto ===
tokenizer = AutoTokenizer.from_pretrained(LORA_REPO)
tokenizer.pad_token = tokenizer.eos_token

# === Carregar modelo base e ajustar os embeddings ===
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
base_model.resize_token_embeddings(len(tokenizer))  # Corrige shape para 50258

# === Merge das LoRA weights (corretamente) ===
merged_model = peft_model.merge_and_unload()

# === Adicionar Value Head ao modelo mergeado ===
model = AutoModelForCausalLMWithValueHead.from_pretrained(merged_model)

# === Salvar modelo final para PPO ===
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


('./modelo_final_para_ppo/tokenizer_config.json',
 './modelo_final_para_ppo/special_tokens_map.json',
 './modelo_final_para_ppo/vocab.json',
 './modelo_final_para_ppo/merges.txt',
 './modelo_final_para_ppo/added_tokens.json',
 './modelo_final_para_ppo/tokenizer.json')

In [9]:
model.push_to_hub(MODEL_HUB)
tokenizer.push_to_hub(MODEL_HUB)

NameError: name 'MODEL_HUB' is not defined

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from trl import AutoModelForCausalLMWithValueHead
# 🔁 Recarregar o modelo já mergeado + value head
from trl import AutoModelForCausalLMWithValueHead
MODEL_HUB = "augustocsc/Se124M100KInfPrompt_EOS_Merged"
#load model
model = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_HUB)
tokenizer = AutoTokenizer.from_pretrained(MODEL_HUB)

# 🔁 Prompt de teste
PROMPT = """
vars: x_1, x_2, x_3, x_4, x_5, x_6, x_7, x_8, x_9, x_10
oper: *, **, +, -, /
cons: C
expr:"""

device = model.pretrained_model.device  # 👈 modelo base dentro do wrapper
input_ids = tokenizer(PROMPT, return_tensors="pt").input_ids.to(device)

# 🔮 Geração
gen_tokens = output = model.generate(
            input_ids=input_ids,
            max_new_tokens=50,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            
        )

# Mostrar resposta
response = tokenizer.decode(gen_tokens[0], skip_special_tokens=False)
print("🧪 Resposta do modelo:\n")
print(response)


Some weights of the model checkpoint at augustocsc/Se124M100KInfPrompt_EOS_Merged were not used when initializing GPT2LMHeadModel: ['v_head.summary.bias', 'v_head.summary.weight']
- This IS expected if you are initializing GPT2LMHeadModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing GPT2LMHeadModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


ValueError: The following `model_kwargs` are not used by the model: ['seed'] (note: typos in the generate arguments will also show up in this list)

In [ ]:
import unittest
import numpy as np
from expression import Expression

class TestExpression(unittest.TestCase):
    def setUp(self):
        # Generate synthetic dataset
        np.random.seed(42)
        self.n_samples = 100
        self.X = np.random.rand(self.n_samples, 5) * 10  # 5 variables, range [0, 10]

    def test_complex_expression(self):
        # Define the complex expression
        goal_expression = "(x_1 + x_2 + C)**C + C*(C*x_3 + C) - C*(x_2 + C) - C*exp(C*x_2 + C)**C"

        # Generate target y using known constants
        real_expression = "(x_1 + x_2 + 2)**2 + 2*(2*x_3 + 2) - 2*(x_2 + 2) - 2*exp(2*x_2 + 2)**2"

        try:
            expr_real = Expression(real_expression)
            y = np.array([expr_real.evaluate(x) for x in self.X])
        except Exception as e:
            self.fail(f"Error evaluating real expression '{real_expression}': {e}")

        try:
            expr_goal = Expression(goal_expression)
            r2 = expr_goal.fit_constants(self.X, y)

            resolved_expr = expr_goal.resolved_expression()
            best_constants = expr_goal.best_constants

            # Assert R^2 is reasonable (close to 1 for a good fit)
            self.assertGreater(r2, 0.9, f"R^2 is too low: {r2}")

            # Print results for debugging
            print(f"Fitted Constants: {best_constants}")
            print(f"Resolved Expression (SymPy): {resolved_expr}")
            print(f"R^2: {r2:.4f}")

        except Exception as e:
            self.fail(f"Error processing goal expression '{goal_expression}': {e}")

if __name__ == "__main__":
    unittest.main()

usage: ipykernel_launcher.py [-h] [-v] [-q] [--locals] [-f] [-c] [-b]
                             [-k TESTNAMEPATTERNS]
                             [tests ...]
ipykernel_launcher.py: error: argument -f/--failfast: ignored explicit argument '/run/user/1010/jupyter/runtime/kernel-v37e0fa52f6bc0dbcb6e40c5e35be4310c1285cc7f.json'


SystemExit: 2

/home/augusto/symbo_repos/seringuela/.seriguela/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3675: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
